In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split,KFold,GridSearchCV
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_selector,make_column_transformer

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features
y = cirrhosis_patient_survival_prediction.data.targets 

In [3]:
X=X.replace('NaNN',np.nan)
X.Cholesterol=pd.to_numeric(X.Cholesterol,errors='coerce')
X.Copper=pd.to_numeric(X.Copper,errors='coerce')
X.Tryglicerides=pd.to_numeric(X.Tryglicerides,errors='coerce')
X.Platelets=pd.to_numeric(X.Platelets,errors='coerce')

In [4]:
imp_cat=SimpleImputer(strategy='most_frequent')
imp_num=SimpleImputer(strategy='mean')
ct_imp=make_column_transformer((imp_cat,make_column_selector(dtype_include=object)),(imp_num, make_column_selector(dtype_exclude= object)),
                                verbose_feature_names_out = False).set_output(transform='pandas')

ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas')

ct_ohe=make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                               ('passthrough',make_column_selector(dtype_exclude=object)),verbose_feature_names_out=False)
ss=StandardScaler()

mm=MinMaxScaler()

kf=KFold(shuffle=True, random_state=26)

rf=RandomForestClassifier(n_estimators=50,random_state=26)

pipe=Pipeline([('IMP',ct_imp),('OHE',ct_ohe),('SC',ss),('RF',rf)])

In [5]:
pipe.get_params()

{'memory': None,
 'steps': [('IMP',
   ColumnTransformer(transformers=[('simpleimputer-1',
                                    SimpleImputer(strategy='most_frequent'),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000210D02B8050>),
                                   ('simpleimputer-2', SimpleImputer(),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000210D02F4690>)],
                     verbose_feature_names_out=False)),
  ('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000210D02F4910>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column

In [7]:
params={
    'IMP__simpleimputer-1__strategy':['constant','most_frequent'],
    'IMP__simpleimputer-2__strategy':[ 'mean','median'],
    'SC':[ss,mm,None],
    'RF__n_estimators': [50,100],
    'RF__max_depth': [1,2,3,4,5,6,None],

}
gcv=GridSearchCV(pipe,param_grid=params,scoring='neg_log_loss',cv=kf,n_jobs=-1)
gcv.fit(X,y)
print('Best params:', gcv.best_params_)
print('Best score:', gcv.best_score_)


C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best params: {'IMP__simpleimputer-1__strategy': 'constant', 'IMP__simpleimputer-2__strategy': 'median', 'RF__max_depth': 5, 'RF__n_estimators': 100, 'SC': None}
Best score: -0.65472669990286
